# Ensembles: bagging, random forests and boosting

One decision tree is unstable (notebook 23). Many trees, combined well, are among the most accurate models on tabular data. There are two ways to combine them. Bagging trains trees independently on resampled data and averages them, which cuts variance. Boosting trains trees one after another, each correcting the last, which cuts bias.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from common.datasets import load_breast_cancer_data, make_noisy_curve

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Ask 100 people to guess the weight of an ox. Each guess is noisy, but the errors point in different directions, so the average lands close to the truth. That only works if the guesses are not all wrong in the same way.

**Bagging** (bootstrap aggregating) makes the guessers different by giving each a different resample of the data. A deep tree has low bias and high variance. Averaging $B$ of them keeps the low bias and shrinks the variance. A **random forest** adds a second source of difference: at each split a tree may only consider a random subset of the features. That stops every tree from picking the same dominant feature first, so the trees are less alike and the average is better.

**Boosting** works the other way. Start with a weak model, such as a tree 2 levels deep, that is too simple on its own (high bias). Look at where it is wrong. Fit the next tree to those mistakes. Add it, with a small weight. Repeat. Each step makes the ensemble a little less biased.

## 2. Maths

**Why averaging helps.** For $B$ models each with variance $\sigma^2$ and pairwise correlation $\rho$, the variance of their average is

$$\operatorname{Var}=\rho\,\sigma^2+\frac{1-\rho}{B}\sigma^2$$

Adding trees kills the second term but not the first. So the goal is to make the trees less correlated, and that is what the random feature subset does. The mean does not change, so bias is unchanged.

**Bootstrap and out-of-bag.** A bootstrap resample draws $n$ rows with replacement. Each row has probability $(1-1/n)^n\to e^{-1}\approx0.368$ of being left out, so about a third of the rows are **out-of-bag** (OOB) for any one tree. Scoring each row only with the trees that never saw it gives a free validation estimate.

**Gradient boosting.** Build a model $F_m=F_{m-1}+\eta\,h_m$ where each new tree $h_m$ is fitted to the negative gradient of the loss at the current predictions. For squared loss $L=\tfrac12(y-F)^2$ the negative gradient is $y-F$, which is the ordinary residual. So each tree is just fitted to what the model still gets wrong. The learning rate $\eta$ shrinks each step. For log-loss the negative gradient is $y-p$, the same quantity that appeared in logistic regression's gradient. This is gradient descent in function space: each tree is a step downhill.

**AdaBoost** is the older version. It keeps a weight per row, increases the weight of misclassified rows after each round, and fits the next weak learner to the reweighted data. The final vote weights each learner by its accuracy.

**Regularisation.** Bagging is already regularised by averaging. For boosting the levers are the learning rate $\eta$, the number of trees, tree depth, row subsampling and early stopping. A smaller $\eta$ needs more trees and generalises better.

## 3. Bagging from scratch

Train 30 unrestricted trees, each on a bootstrap resample of the training data, and take a majority vote. Compare it against one tree, on held-out data, over 12 different splits so the comparison is not a fluke of one split.

In [ ]:
Xb, yb = load_breast_cancer_data()


def bagged_predict(X_tr, y_tr, X_te, n_trees: int = 100, seed: int = 0) -> np.ndarray:
    gen = np.random.default_rng(seed)
    votes = np.zeros(len(X_te))
    for _ in range(n_trees):
        idx = gen.choice(len(X_tr), size=len(X_tr), replace=True)
        tree = DecisionTreeClassifier(random_state=int(gen.integers(1 << 30)))
        votes += tree.fit(X_tr[idx], y_tr[idx]).predict(X_te)
    return (votes / n_trees > 0.5).astype(int)


single, bagged = [], []
for s in range(12):
    a, b, c, d = train_test_split(
        Xb.to_numpy(), yb.to_numpy(), test_size=0.3, random_state=s, stratify=yb
    )
    single.append((DecisionTreeClassifier(random_state=s).fit(a, c).predict(b) == d).mean())
    bagged.append((bagged_predict(a, c, b, n_trees=30, seed=s) == d).mean())
print(f"single tree: mean {np.mean(single):.3f}  sd {np.std(single):.3f}")
print(f"bagged 30:   mean {np.mean(bagged):.3f}  sd {np.std(bagged):.3f}")

Averaging lifts the accuracy by about three points and also shrinks the spread between splits. That is variance reduction at work.

## 4. Random forest

`RandomForestClassifier` is bagging plus the feature subset (`max_features`, default $\sqrt{p}$ for classification). The OOB score comes free with `oob_score=True`.

In [ ]:
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=SEED, stratify=yb)
rows = []
for n in (1, 5, 20, 100, 300):
    rf = RandomForestClassifier(n_estimators=n, oob_score=n >= 20, random_state=SEED, n_jobs=1).fit(
        Xb_tr, yb_tr
    )
    rows.append(
        {
            "trees": n,
            "test acc": rf.score(Xb_te, yb_te),
            "OOB acc": rf.oob_score_ if n >= 20 else np.nan,
        }
    )
pd.DataFrame(rows).set_index("trees").round(3)

More trees help quickly and then flatten. A forest does not overfit by adding trees, because it is an average, so you can set `n_estimators` high and stop when the curve is flat. The OOB estimate lands within about a point of the test score without touching the test set.

The feature subset matters. Compare `max_features` from a single feature per split to all of them (which is plain bagging).

In [ ]:
rows = []
for mf in (1, "sqrt", 15, 30):
    r = cross_validate(RandomForestClassifier(50, max_features=mf, random_state=SEED), Xb, yb, cv=5)
    rows.append({"max_features": mf, "cv acc": r["test_score"].mean()})
pd.DataFrame(rows).set_index("max_features").round(4)

With 30 features the five-fold accuracy is the same (0.956) for almost every setting, because the breast cancer data are easy and several features carry the same signal. The default of $\sqrt{30}\approx5$ is a sound starting point. The effect is bigger when one feature dominates.

## 5. Boosting from scratch: residual fitting

Gradient boosting for squared loss in 10 lines. Start from the mean, then repeatedly fit a shallow tree to the residuals and add a fraction $\eta$ of it. It is easiest to see on the noisy curve from notebook 11.

In [ ]:
X_c, y_c = make_noisy_curve(n_samples=80, noise=0.3, seed=SEED)
grid = pd.DataFrame({"x": np.linspace(-3, 3, 300)})


def boost_fit(X, y, n_trees: int = 100, lr: float = 0.1, depth: int = 2):
    base = y.mean()
    pred = np.full(len(y), base)
    trees = []
    for _ in range(n_trees):
        resid = y - pred  # the negative gradient of squared loss
        t = DecisionTreeRegressor(max_depth=depth, random_state=SEED).fit(X, resid)
        pred = pred + lr * t.predict(X)
        trees.append(t)
    return base, trees


def boost_predict(model, X, lr: float = 0.1, n: int | None = None) -> np.ndarray:
    base, trees = model
    return base + lr * sum(t.predict(X) for t in trees[:n])


model = boost_fit(X_c, y_c.to_numpy())
fig, axes = plt.subplots(1, 4, figsize=(15, 3), sharey=True)
for ax, n in zip(axes, (1, 5, 30, 100)):
    ax.scatter(X_c["x"], y_c, s=10, c="grey")
    ax.plot(grid["x"], boost_predict(model, grid, n=n), c="tab:red")
    ax.set_title(f"{n} trees")
plt.show()
ref = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, max_depth=2, random_state=SEED
).fit(X_c, y_c)
print(
    "max difference from sklearn:",
    np.abs(boost_predict(model, X_c) - ref.predict(X_c)).max().round(4),
)

One tree gives a crude staircase (high bias). Each added tree fits what is left and the curve bends towards the data. By 100 trees it follows the wave. Keep going and it would start chasing the noise, which is why the number of trees and the learning rate need tuning. The predictions match sklearn's `GradientBoostingRegressor` to four decimals.

**AdaBoost** reaches the same end by reweighting rows instead of fitting residuals. It is in the comparison below.

## 6. Tuning gradient boosting

The three main dials are the number of trees, the learning rate and the tree depth. The learning rate and number of trees trade against each other. Track the test log-loss after each added tree for three learning rates.

In [ ]:
from sklearn.metrics import log_loss

fig, ax = plt.subplots()
for lr in (0.5, 0.1, 0.02):
    gb = GradientBoostingClassifier(
        n_estimators=300, learning_rate=lr, max_depth=2, random_state=SEED
    ).fit(Xb_tr, yb_tr)
    curve = [log_loss(yb_te, p) for p in gb.staged_predict_proba(Xb_te)]
    ax.plot(curve, label=f"lr={lr}")
    print(f"lr={lr}: best at {int(np.argmin(curve)) + 1} trees, test log-loss {min(curve):.3f}")
ax.set_xlabel("trees")
ax.set_ylabel("test log-loss")
ax.set_ylim(0, 0.6)
ax.legend()
plt.show()

The large rate (0.5) bottoms out after 31 trees and then climbs as it overfits. The middle rate (0.1) bottoms after 137 trees. The small rate (0.02) is still improving at 300 trees. On this small clean set all three reach a similar best loss (0.08 to 0.09), so the benefit of a small rate is not visible. On bigger and noisier data it usually is, at the price of more trees. The usual recipe is a small $\eta$ such as 0.05 to 0.1, many trees, and **early stopping**: hold out part of the training data and stop when its score stops improving. `GradientBoostingClassifier(n_iter_no_change=10, validation_fraction=0.2)` and `HistGradientBoostingClassifier(early_stopping=True)` do this. Shallow trees (depth 2 to 4) and `subsample=0.7` also regularise.

In [ ]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.1, max_iter=500, early_stopping=True, n_iter_no_change=10, random_state=SEED
).fit(Xb_tr, yb_tr)
print("trees used before stopping:", hgb.n_iter_, " test acc:", round(hgb.score(Xb_te, yb_te), 3))

## 7. Comparison on breast cancer

Every model so far, with scaling where it is needed, on the same 5-fold cross-validation. I report accuracy, AUC (notebook 25 explains it) and fit time.

In [ ]:
models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "kNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "SVM (rbf)": make_pipeline(StandardScaler(), SVC()),
    "decision tree": DecisionTreeClassifier(random_state=SEED),
    "tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=SEED),
    "random forest": RandomForestClassifier(200, random_state=SEED),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=SEED),
    "gradient boosting": GradientBoostingClassifier(random_state=SEED),
    "hist gradient boosting": HistGradientBoostingClassifier(random_state=SEED),
}
rows = []
for name, m in models.items():
    r = cross_validate(m, Xb, yb, cv=5, scoring=["accuracy", "roc_auc"])
    rows.append(
        {
            "model": name,
            "accuracy": r["test_accuracy"].mean(),
            "AUC": r["test_roc_auc"].mean(),
            "fit s": r["fit_time"].mean(),
        }
    )
pd.DataFrame(rows).set_index("model").round(3).sort_values("AUC", ascending=False)

On this small, clean dataset the scaled logistic regression (0.981) and SVM (AUC 0.996) are as good as the best ensembles (AdaBoost at 0.977). The single trees are last by some distance, at about 0.92, and every ensemble beats them comfortably. Ensembles earn their place on larger, messier tabular data with interactions and mixed types, where linear models struggle. A 569-row, 30-feature set with a nearly linear boundary is not that case, which is a useful reminder: always run the simple baseline first.

## 8. What breaks

- **Interpretability.** 200 trees cannot be read. Use feature importances or permutation importance.
- **Noisy labels.** Boosting keeps fitting the mistakes, and so fits mislabelled rows too. Bagging is more forgiving. If labels are noisy, use a small learning rate, shallow trees and early stopping.
- **Extrapolation.** Trees predict inside the range of the training targets. Neither a forest nor boosting can continue a trend.
- **Cost.** Forests parallelise well. Boosting is sequential, and gets slower with more trees and lower learning rates. `HistGradientBoosting` bins features and is the fast default for large data.
- **Leakage and tuning.** With flexible models, tune on a validation split or by CV, not on the test set (notebook 00).

Next, how to judge a classifier properly, because accuracy alone hides a great deal.